# Week 8 Exercise – The Price is Right: Deal Agent Framework & UI

This notebook runs the Week 8 **Deal Agent Framework**: Scanner → Ensemble (Frontier + Specialist + Neural Network) → Planning Agent → Messaging. It uses the course's Chroma vector store, Modal pricer (if deployed), and optional Gradio UI with deal table and row-click alerts.

**Prerequisites:** Week 8 setup complete (Chroma `products_vectorstore`, optional Modal pricer, optional Pushover for alerts). Run from repo root or ensure `week8` is on `PYTHONPATH`; the notebook switches working directory to `week8` so relative paths work.

## Path and working directory

Set up the Python path and change to `week8` so `deal_agent_framework`, `agents`, and `log_utils` resolve and so `memory.json` and `products_vectorstore` are found.

In [ ]:
import os
import sys
from pathlib import Path


notebook_dir = Path.cwd()
week8_dir = notebook_dir.parent.parent
if (week8_dir / "deal_agent_framework.py").exists():
    os.chdir(week8_dir)
    sys.path.insert(0, str(week8_dir))
    print(f"Working directory: {os.getcwd()}")
else:
    week8_dir = notebook_dir / "week8" if (notebook_dir / "week8").exists() else notebook_dir
    if (week8_dir / "deal_agent_framework.py").exists():
        os.chdir(week8_dir)
        sys.path.insert(0, str(week8_dir))
        print(f"Working directory: {os.getcwd()}")
    else:
        print("Warning: deal_agent_framework.py not found; ensure you run from repo or week8/community_contributions/victor_barny")

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

## Initialize the Deal Agent Framework

Load Chroma, read `memory.json`, and lazily create the Planning Agent (Scanner, Ensemble, Messaging).

In [ ]:
import logging
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

logging.getLogger().setLevel(logging.INFO)

framework = DealAgentFramework()
framework.init_agents_as_needed()
print(f"Memory: {len(framework.memory)} opportunity/opportunities")

## Run one planning cycle

Scan RSS, price top deals with the ensemble, pick the best above threshold, optionally alert via MessagingAgent. New opportunity is appended to memory and saved to `memory.json`.

In [ ]:
opportunities = framework.run()
print(f"Total opportunities in memory: {len(opportunities)}")
for i, opp in enumerate(opportunities[-3:], start=max(1, len(opportunities) - 2)):
    print(f"  {i}. discount ${opp.discount:.2f} | {opp.deal.product_description[:50]}...")

## Table view of opportunities

Helper to render the current memory as a list of rows for a Gradio Dataframe.

In [ ]:
def get_table(opps):
    return [
        [
            opp.deal.product_description,
            opp.deal.price,
            opp.estimate,
            opp.discount,
            opp.deal.url,
        ]
        for opp in opps
    ]

if framework.memory:
    for row in get_table(framework.memory):
        print(row)
else:
    print("No opportunities in memory yet. Run a cycle (and ensure Chroma/Modal/RSS are set up).")

## Optional: Gradio UI

Launch a small UI: table of opportunities (from framework memory), load on startup, and row selection triggers the MessagingAgent alert for that opportunity.

In [ ]:
import gradio as gr

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    opportunities_state = gr.State(framework.memory)

    def get_table_from_state(opps):
        return get_table(opps)

    def do_select(opps, selected_index: gr.SelectData):
        if not opps:
            return
        row = selected_index.index[0]
        if row < len(opps):
            framework.planner.messenger.alert(opps[row])

    gr.Markdown('<div style="text-align: center; font-size: 24px;">The Price is Right – Victor Barny</div>')
    gr.Markdown('<div style="text-align: center; font-size: 14px;">Deals surfaced so far (select a row to send alert)</div>')
    opportunities_dataframe = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        wrap=True,
        column_widths=[4, 1, 1, 1, 2],
        row_count=10,
        col_count=5,
        max_height=400,
    )

    ui.load(get_table_from_state, inputs=[opportunities_state], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities_state], outputs=[])

ui.launch(inbrowser=True)